---
## 🎁 가산점

### A. 데이터의 다양성
- NTP ICE 내 다양한 데이터셋 모두 활용 가능. (https://ice.ntp.niehs.nih.gov/DATASETDESCRIPTION)
### B. Feature(descriptor)의 다양성
- rdkit, VEGA, 등
### 💬 추가 설명 (자유 기술)

# 기말고사 Template 1 — Data Pipeline

**이름:** ___육건우___ &nbsp; **학번:** _____20251284_____ &nbsp;

---

## 📋 채점 기준 (총 50점)

| 항목 | 배점 | 채점 포인트 |
|---|---|---|
| **1. 데이터 분포 파악 및 전처리** | 15점 | 모델 개발 전, 중복 화합물 체크, smiles 코드 정리 등 모델 개발 전 확인해야 할 사항들을 확인. |
| **2. Descriptor 계산** | 15점 | 모델 개발에 사용할 descriptor의 다양성 |
| **3. 데이터 시각화 자료** | 15점 | 구조 분포, 라벨 비율 등 데이터 현황을 시각화한 자료 |
| **4. 코드 가독성 & 주석** | 5점 | 변수의 의미와 코드의 간결성을 평가. |

#### A. 데이터 소스의 다양성
- NTP ICE에서 구할 수 있는 다양한 데이터
- NTP ICE 외 추가 데이터 확보

## 📁 입력 / 출력 예시
- **입력**: `skin_irritation.xlsx` (NTP ICE) + (선택) 외부 데이터
- **출력**: `final_dataset_descriptors.csv`  (Chemical_Name, SMILES, label, 2D descriptor [+ fingerprint 등])

나의 탐구 주제는 "새 화학물질의 발암 가능성을 자동 예측하는 모델 만들기"

연구 배경: 화학 물질의 안전성 평가 및 동물 실험 대체 연구의 필요성
연구 목적: SMILES 구조 데이터를 활용한 머신러닝 기반 발암성(Carcinogenicity) 예측 QSAR 모델 구축

In [20]:
import pandas as pd

file_path = 'cancer.xlsx' 
sheet_all = 'Data'

# pd.read_excel 함수를 사용해 특정 시트를 읽어온다.
df = pd.read_excel(file_path, sheet_name=sheet_all)

print(df.shape) #전체 데이터의 크기 행과 열의 개수 확인
print(df.columns) #전체 컬럼 이름 확인

(10351, 28)
Index(['Record_ID', 'Data_Type', 'Formulation_ID', 'Formulation_Name',
       'Chemical_Name', 'CASRN', 'DTXSID', 'Percent_Active_Ingredient',
       'Mixture', 'Species', 'Sex', 'Strain', 'Route', 'Level_of_Evidence',
       'Tissue', 'Location', 'Lesion', 'Assay', 'Endpoint', 'Response',
       'Response_Unit', 'Reference', 'URL', 'SMILES', 'Preferred_Name',
       'Synonyms', 'URL_CompTox', 'URL_CEBS'],
      dtype='object')


In [21]:
# 전체 데이터 중 'NTP Carcinogenicity' 관련 데이터만 골라내기
is_ntp = df["Endpoint"] == "Level of evidence of carcinogenic activity"
ntp_df = df[is_ntp].copy()

# 그중에서 생체 내(In Vivo) 실험 데이터만 남기기
ntp_in_vivo_df = ntp_df[ntp_df["Data_Type"] == "In Vivo"].copy()

# 분자 구조(SMILES)가 비어있는 행은 제외하기
ntp_in_vivo_df = ntp_in_vivo_df[ntp_in_vivo_df["SMILES"].notna()].copy()

# 현재까지 정제된 데이터의 크기 확인
ntp_in_vivo_df.shape

(2213, 28)

In [22]:
# NTP Carcinogenicity + In Vivo 조건 만족하는 데이터의 종별 개수 확인
species_counts_final = ntp_in_vivo_df["Species"].value_counts()

# 결과 확인하기
species_counts_final

Species
Rat        1103
Mouse      1102
Hamster       8
Name: count, dtype: int64

In [23]:
#종에 따른 respone 데이터 개수 확인
pd.crosstab(
    ntp_in_vivo_df["Species"],
    ntp_in_vivo_df["Response"]
)

Response,Clear evidence,Equivocal evidence,Inadequate experiment,No evidence,Not tested in sex/species,Some evidence
Species,,,,,,
Hamster,0,0,0,6,0,2
Mouse,313,89,24,516,99,61
Rat,277,115,26,503,95,87


In [24]:
#종별로 smiles 코드가 중복되는 것이 많다면 종을 선택하지 않고 프로젝트를 진행한 후 어떤 종에서 더 정확성을 보이는지 확인.
# 겹치는 smiles 코드가 없다면 종을 선택한 다음 1가지 종에 몰두하여 프로젝트 진행하겠음

# 전체 행 수
print("전체 행 수:", len(ntp_in_vivo_df))

# 고유 SMILES 개수
print("고유 SMILES 수:", ntp_in_vivo_df["SMILES"].nunique())

# 중복 SMILES 개수
print(
    "중복된 SMILES 수:",
    len(ntp_in_vivo_df) - ntp_in_vivo_df["SMILES"].nunique()
)

전체 행 수: 2213
고유 SMILES 수: 504
중복된 SMILES 수: 1709


In [25]:
#실제로 어떤 SMILES가 중복되는지 확인 - 2번 이상 등장한 스마일즈 종류 수
smiles_counts = ntp_in_vivo_df["SMILES"].value_counts()

duplicated_smiles = smiles_counts[smiles_counts > 1]

print("중복된 SMILES 종류 수:", len(duplicated_smiles))

duplicated_smiles.head(20)

중복된 SMILES 종류 수: 504


SMILES
CC1=CN([C@H]2C[C@H](N=[N+]=[N-])[C@@H](CO)O2)C(=O)NC1=O                                                   24
O.[Mg++].[Mg++].[Mg++].O[Si]([O-])([O-])[O-].O[Si]([O-])([O-])[O-]                                        18
ClC1=C(Cl)C=C2OC3=CC(Cl)=C(Cl)C=C3OC2=C1                                                                  16
ClC1=CC(=CC(Cl)=C1Cl)C1=CC(Cl)=C(Cl)C=C1                                                                  16
ClC=C(Cl)Cl                                                                                               12
OC1=C(Cl)C(Cl)=C(Cl)C(Cl)=C1Cl                                                                            12
NC1=NC(=O)N(C=C1)[C@@H]1CS[C@H](CO)O1                                                                     12
OC1N=C(C2=CC=CC=C2)C2=CC(Cl)=CC=C2NC1=O                                                                   10
Cl.Cl.CNNC                                                                                                10
ClCC(Br)CBr 

고유 SMILES 수 = 504
중복된 SMILES 종류 수 = 504
모든 SMILES가 최소 2번 이상 등장했다는 의미이다.

In [31]:
test = ntp_in_vivo_df.groupby("SMILES").agg({
    "Species": "nunique",
    "Sex": "nunique",
    "Response": "nunique"
})

test.head()

test.describe()

,Species,Sex,Response
count,504.000000,504.0,504.000000
mean,2.001984,2.0,1.914683
std,0.077203,0.0,0.852235
min,1.000000,2.0,1.000000
25%,2.000000,2.0,1.000000
50%,2.000000,2.0,2.000000
75%,2.000000,2.0,2.000000
max,3.000000,2.0,6.000000


In [27]:
species_per_smiles = (
    ntp_in_vivo_df
    .groupby("SMILES")["Species"]
    .nunique()
)

both_species = species_per_smiles[species_per_smiles > 1]

print("Mouse와 Rat 모두에 존재하는 SMILES 수:")
print(len(both_species))

Mouse와 Rat 모두에 존재하는 SMILES 수:
503


In [28]:
shared_smiles = both_species.index

ntp_in_vivo_df[
    ntp_in_vivo_df["SMILES"].isin(shared_smiles)
].sort_values(["SMILES", "Species"]).head(30)

,Record_ID,Data_Type,Formulation_ID,Formulation_Name,Chemical_Name,CASRN,DTXSID,Percent_Active_Ingredient,Mixture,Species,...,Endpoint,Response,Response_Unit,Reference,URL,SMILES,Preferred_Name,Synonyms,URL_CompTox,URL_CEBS
2901,cancer_6589,In Vivo,NaN,Divinylbenzene-HP,Divinylbenzene,1321-74-0,DTXSID2025216,NaN,Chemical,Mouse,...,Level of evidence of carcinogenic activity,No evidence,Unitless,TR-534,https://ntp.niehs.nih.gov/publications/reports...,"*C=C.C=CC1=CC=CC=C1 |c:6,8,t:4,m:0:8.6|",Divinylbenzene,"1321-74-0|Divinylbenzene|Benzene, diethenyl-|B...",https://comptox.epa.gov/dashboard/chemical/det...,https://doi.org/10.22427/NTP-DATA-DTXSID2025216
2909,cancer_6594,In Vivo,NaN,Divinylbenzene-HP,Divinylbenzene,1321-74-0,DTXSID2025216,NaN,Chemical,Mouse,...,Level of evidence of carcinogenic activity,Equivocal evidence,Unitless,TR-534,https://ntp.niehs.nih.gov/publications/reports...,"*C=C.C=CC1=CC=CC=C1 |c:6,8,t:4,m:0:8.6|",Divinylbenzene,"1321-74-0|Divinylbenzene|Benzene, diethenyl-|B...",https://comptox.epa.gov/dashboard/chemical/det...,https://doi.org/10.22427/NTP-DATA-DTXSID2025216
2904,cancer_6597,In Vivo,NaN,Divinylbenzene-HP,Divinylbenzene,1321-74-0,DTXSID2025216,NaN,Chemical,Rat,...,Level of evidence of carcinogenic activity,No evidence,Unitless,TR-534,https://ntp.niehs.nih.gov/publications/reports...,"*C=C.C=CC1=CC=CC=C1 |c:6,8,t:4,m:0:8.6|",Divinylbenzene,"1321-74-0|Divinylbenzene|Benzene, diethenyl-|B...",https://comptox.epa.gov/dashboard/chemical/det...,https://doi.org/10.22427/NTP-DATA-DTXSID2025216
2914,cancer_6595,In Vivo,NaN,Divinylbenzene-HP,Divinylbenzene,1321-74-0,DTXSID2025216,NaN,Chemical,Rat,...,Level of evidence of carcinogenic activity,Equivocal evidence,Unitless,TR-534,https://ntp.niehs.nih.gov/publications/reports...,"*C=C.C=CC1=CC=CC=C1 |c:6,8,t:4,m:0:8.6|",Divinylbenzene,"1321-74-0|Divinylbenzene|Benzene, diethenyl-|B...",https://comptox.epa.gov/dashboard/chemical/det...,https://doi.org/10.22427/NTP-DATA-DTXSID2025216
1611,cancer_3704,In Vivo,NaN,Tribromomethane,Bromoform,75-25-2,DTXSID1021374,NaN,Chemical,Mouse,...,Level of evidence of carcinogenic activity,No evidence,Unitless,TR-350,https://ntp.niehs.nih.gov/publications/reports...,BrC(Br)Br,Bromoform,75-25-2|Bromoform|4-01-00-00082|BRN 1731048|br...,https://comptox.epa.gov/dashboard/chemical/det...,https://doi.org/10.22427/NTP-DATA-DTXSID1021374
1614,cancer_3705,In Vivo,NaN,Tribromomethane,Bromoform,75-25-2,DTXSID1021374,NaN,Chemical,Mouse,...,Level of evidence of carcinogenic activity,No evidence,Unitless,TR-350,https://ntp.niehs.nih.gov/publications/reports...,BrC(Br)Br,Bromoform,75-25-2|Bromoform|4-01-00-00082|BRN 1731048|br...,https://comptox.epa.gov/dashboard/chemical/det...,https://doi.org/10.22427/NTP-DATA-DTXSID1021374
1608,cancer_3703,In Vivo,NaN,Tribromomethane,Bromoform,75-25-2,DTXSID1021374,NaN,Chemical,Rat,...,Level of evidence of carcinogenic activity,Some evidence,Unitless,TR-350,https://ntp.niehs.nih.gov/publications/reports...,BrC(Br)Br,Bromoform,75-25-2|Bromoform|4-01-00-00082|BRN 1731048|br...,https://comptox.epa.gov/dashboard/chemical/det...,https://doi.org/10.22427/NTP-DATA-DTXSID1021374
1622,cancer_3710,In Vivo,NaN,Tribromomethane,Bromoform,75-25-2,DTXSID1021374,NaN,Chemical,Rat,...,Level of evidence of carcinogenic activity,Clear evidence,Unitless,TR-350,https://ntp.niehs.nih.gov/publications/reports...,BrC(Br)Br,Bromoform,75-25-2|Bromoform|4-01-00-00082|BRN 1731048|br...,https://comptox.epa.gov/dashboard/chemical/det...,https://doi.org/10.22427/NTP-DATA-DTXSID1021374
3852,cancer_1266,In Vivo,NaN,Dibromoacetonitrile,Dibromoacetonitrile,3252-43-5,DTXSID3024940,NaN,Chemical,Mouse,...,Level of evidence of carcinogenic activity,Clear evidence,Unitless,TR-544,https://ntp.niehs.nih.gov/publications/reports...,BrC(Br)C#N,Dibromoacetonitrile,3252-43-5|Dibromoacetonitrile|4-02-00-00533|Ac...,https://comptox.epa.gov/dashboard/chemical/det...,https://doi.org/10.22427/NTP-DATA-DTXSID3024940
3857,cancer_1269,In Vivo,NaN,Dibromoaceton

In [26]:
#성별 분포에 불균형이 있지 않은지 확인. 만약 있다면 성별에 따른 발암성이 다를 수 있으니 고려할 계획
#결과 확인 -- 비율이 비슷한 관계로 따로 신경쓰지 않기로 결정

# 랫(Rat) 데이터의 성별 구성 확인하기
rat_sex = ntp_in_vivo_df[ntp_in_vivo_df["Species"] == "Rat"]["Sex"].value_counts()
print("--- 랫(Rat) 성별 분포 ---")
print(rat_sex)

# 마우스(Mouse) 데이터의 성별 구성 확인하기
mouse_sex = ntp_in_vivo_df[ntp_in_vivo_df["Species"] == "Mouse"][
    "Sex"
].value_counts()
print("--- 마우스(Mouse) 성별 분포 ---")
print(mouse_sex)

--- 랫(Rat) 성별 분포 ---
Sex
Female    552
Male      551
Name: count, dtype: int64
--- 마우스(Mouse) 성별 분포 ---
Sex
Female    551
Male      551
Name: count, dtype: int64
